In [2]:
"""
Parse ANADDB output file
Currently able to parse: EO tensor, Raman tensor, polarities.
"""
from itertools import tee
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt

### Reading the file ###
class Anaddb:
    def __init__(self, filename, natom=10):
        with open(filename, 'r') as file:
            self.data = file.readlines()
        self.natom = natom
        
    def ionic_EO(self):
        ionic_EO_tensor = np.zeros((6,3))
        index = self.data.index(" Total EO tensor (pm/V) in Voigt notations\n")+1
        
        for i in range(6):
            ionic_EO_tensor[i] = np.array(self.data[index+i].split(), dtype=float)
            
        return ionic_EO_tensor
    
    def get_EO_tensor(self):
        EO_tensors = pd.DataFrame(columns=["Mode number", "Frequency", "EO tensor"])
        index = self.data.index(" Output of the EO tensor (pm/V) in Voigt notations\n")+6
        
        for mode_number in range(4, (self.natom*3) + 1):
            ### space + digit + . + digit?
            frequency = float(re.findall("\s\d+.\d+", self.data[index])[0])
            
            EO = np.zeros((6, 3))
            for i in range(6):
                EO[i] = np.array(self.data[index+i+1].split(), dtype = float)
                
            EO_tensors.loc[len(EO_tensors.index)] = [mode_number, frequency, EO]
            index +=8
        return EO_tensors
            

    def get_Raman(self, NAC=False):
        Raman_tensors = pd.DataFrame(columns=["Mode", "Freq", "Raman susceptibility"])

        if NAC:
            index = self.data.index(" Raman susceptibility of zone-center phonons, with non-analyticity in the\n")+23
        else:
            index = self.data.index("  Raman susceptibilities of transverse zone-center phonon modes\n")+22

        for mode in np.arange(4, self.natom*3+1):
            freq = float(re.findall("\s\d+.\d+", self.data[index])[0])

            Raman = np.zeros((3,3))
            for i in range(3):
                Raman[i] = np.array(self.data[index+i+1][1:].split(), dtype=float)

            Raman_tensors.loc[len(Raman_tensors.index)] = [mode, freq, Raman]
            index += 6

        return Raman_tensors


    def get_polarity(self):
        polarity = pd.DataFrame(columns=["Mode", "px", "py", "pz", "|p|"])
        index = self.data.index(" Mode effective charges \n")+5
        
        for mode in np.arange(4, self.natom*3+1):
            polarity.loc[len(polarity.index)] = self.data[index][1:].split()
            index += 1

        return polarity
    
    
    def get_disp_mag(self):
        displacement = pd.DataFrame(columns=["Mode", "displacement set"])
        index = self.data.index(" Treat the first list of vectors \n")+93
        
        for mode in range(4, self.natom*3 + 1):
            disp = np.zeros((10, 3))
            
            for i in range(10):
                disp[i] = np.array(self.data[index+2*i][6:].split(), dtype=float) 
            
            displacement.loc[len(displacement.index)] = [mode, disp]
            index += 21
            
        return displacement
    
    def get_BEC(self):
        born_ec = pd.DataFrame(columns=[])

    

In [3]:
if __name__ == "__main__":
    LNO = Anaddb("tnlo_4.abo")
    print(LNO.ionic_EO()*(-1))

[[-0.         -4.78801885 10.07651212]
 [-0.          4.78801885 10.07651212]
 [ 0.         -0.         27.6903241 ]
 [ 0.         15.36850141 -0.        ]
 [15.36850141  0.          0.        ]
 [-4.78801885 -0.          0.        ]]


In [101]:
a = LNO.get_disp_mag().loc[0, 'displacement set']
b = a.transpose()
np.savetxt(sys.stdout, b)


1.837187520000000090e-04 1.840575700000000024e-04 5.075751469999999731e-04 5.073613089999999678e-04 7.453438759999999512e-04 1.170029379999999947e-03 1.110665150000000097e-03 7.457125039999999835e-04 1.110951430000000012e-03 1.170450030000000060e-03
-1.150073269999999943e-03 1.150019099999999989e-03 7.257532239999999607e-04 -7.259027309999999556e-04 1.182791209999999905e-03 1.006146830000000016e-03 1.462257459999999907e-03 -1.182416410000000032e-03 -1.461990270000000040e-03 -1.005897150000000006e-03
0.000000000000000000e+00 0.000000000000000000e+00 0.000000000000000000e+00 0.000000000000000000e+00 -1.492492310000000056e-03 5.639595010000000164e-05 1.436096360000000067e-03 1.492257560000000019e-03 -1.436359749999999939e-03 -5.589781009999999763e-05


In [90]:
a[:][0:1]

array([[ 0.00018372, -0.00115007,  0.        ]])

In [96]:
disp_amp = []
for i in range(27):
    a = LNO.get_disp_mag().loc[i, 'displacement set']
    disp_amp_mode = []
    for i in range(10):
        x = np.linalg.norm(a[:][i:i+1])
        disp_amp_mode.append(x)
    disp_amp.append(disp_amp_mode)    
    
disp_amp

    

[[0.0011646549300994395,
  0.00116465493578103,
  0.0008856355187087212,
  0.0008856355191293152,
  0.00204500998416954,
  0.0015441763813664276,
  0.002331125631525556,
  0.0020447563010384372,
  0.0023312567342130682,
  0.0015443143832390798],
 [0.0011646549300994395,
  0.00116465493578103,
  0.0008856355187087212,
  0.0008856355191293152,
  0.001954071732383123,
  0.0023698062077239894,
  0.001601972363794601,
  0.001954337185986765,
  0.0016017815655872986,
  0.0023697162751495004],
 [0.00404971504,
  0.00404971504,
  0.0011419903,
  0.0011419903,
  0.0009047858710101638,
  0.0009047858741057492,
  0.0009047858757208778,
  0.0009047858710101638,
  0.0009047858757208778,
  0.0009047858741057492],
 [0.001668123718725631,
  0.0016681237162633291,
  0.0011139502447928707,
  0.0011139502427450713,
  0.0019322554723739114,
  0.000915881073224856,
  0.002039233549886303,
  0.0019321440087362272,
  0.0020393256337715377,
  0.0009159111962164947],
 [0.001668123718725631,
  0.001668123716263

In [20]:
polarity = LNO.get_polarity()
b = polarity.loc[:, "|p|"]
polarity

,Mode,px,py,pz,|p|
0,4,4.629780,-0.000000,-0.000000,4.629780
1,5,0.000000,4.629780,-0.000000,4.629780
2,6,0.000000,0.000000,-0.000000,0.000000
3,7,-1.768584,-0.000000,-0.000000,1.768584
4,8,-0.000000,-1.768584,-0.000000,1.768584
5,9,-0.000000,-0.000000,6.900053,6.900053
6,10,-3.591195,-0.000000,-0.000000,3.591195
7,11,-0.000000,-3.591195,-0.000000,3.591195
8,12,-0.000000,-0.000000,0.239492,0.239492
9,13,-0.000000,0.000000,-0.000000,0.000000


In [80]:
raman = LNO.get_Raman()
c = raman.loc[:, "Raman susceptibility"]

max_comp = []
for i in range(27):
    max_comp.append(np.max(np.max(c[i:i+1])))
np.savetxt(sys.stdout, max_comp)


4.836354999999999647e-03
4.836354999999999647e-03
-0.000000000000000000e+00
1.688714999999999921e-03
1.688714999999999921e-03
1.981410499999999852e-02
1.394655999999999977e-03
1.394655999999999977e-03
-0.000000000000000000e+00
-0.000000000000000000e+00
0.000000000000000000e+00
5.378319000000000329e-03
6.788879999999999958e-04
2.207844000000000039e-03
2.207844000000000039e-03
2.236501000000000114e-03
2.236501000000000114e-03
-0.000000000000000000e+00
4.548634000000000184e-03
4.548634000000000184e-03
-0.000000000000000000e+00
4.610420999999999964e-03
4.610420999999999964e-03
3.186334899999999914e-02
2.128433999999999795e-03
2.128433999999999795e-03
0.000000000000000000e+00


In [82]:
frequency = LNO.get_EO_tensor()
d = frequency.loc[:, "Frequency"]
d

0     152.04
1     152.04
2     215.46
3     215.75
4     215.75
5     239.43
6     261.74
7     261.74
8     284.69
9     296.66
10    335.15
11    335.15
12    358.52
13    377.83
14    377.83
15    389.86
16    389.86
17    414.82
18    433.55
19    433.55
20    455.93
21    581.74
22    581.74
23    608.90
24    671.58
25    671.58
26    892.15
Name: Frequency, dtype: float64